# Uber Assignment Class

###  Bus 36109 "Advanced Decision Modeling with Python", Don Eisenstein
Don Eisenstein &copy; Copyright 2023, University of Chicago 

---

In [1]:
from pulp import *

# Portable solver setup, to allow work locally (ARM64 architecture) as well as in JupyterHub
import os
from pulp import COIN_CMD
if os.path.exists('/opt/homebrew/opt/cbc/bin/cbc'):
    solver = COIN_CMD(path='/opt/homebrew/opt/cbc/bin/cbc', msg=0)
else:
    solver = pulp.PULP_CBC_CMD(msg=0)

In [2]:
class Uber:
    """
     Uber Class Documentation: 

         Usage:
             uber = Uber()  
             # Create a model instance

             uber.set_drivers(drivers)  
             # provide a list of driver name strings
             # Example: uber.set_drivers(['D32', 'D8712', 'D922', 'D88'])

             uber.set_customers(customers)  
             # provide a list of customer name strings
             # Example: uber.set_customers(['C11', 'C934', 'C331'])

             uber.set_travel_times()
             # provide a dictionary of travel times between all Driver/Customer pairs
             # Example:  uber.set_travel_times({('D32', 'C11'): 587, 
                 ('D32', 'C331'): 1380,
                 ('D32', 'C934'): 915,
                 ('D8712', 'C11'): 1734, ... })

             uber.build_model()  
             # initialize optimization model

             uber.print_model()
             # print model objective and constraints

             uber.solve_model()
             # Solve the model

             uber.status()
             # Return model status, 'Optimal', 'Infeasible', or 'Unbounded'

             uber.objective_function_value()
             # Return objective function value

             uber.solution()
             # Return list of all-pairs solution
             # Example:  [['D32', 'C11', 1.0],
                 ['D32', 'C331', 0.0],
                 ['D32', 'C934', 0.0],
                 ['D8712', 'C11', 0.0],
                 ['D8712', 'C331', 0.0],
                 ['D8712', 'C934', 0.0],
                 ['D88', 'C11', 0.0],
                 ['D88', 'C331', 0.0],
                 ['D88', 'C934', 1.0],
                 ['D922', 'C11', 0.0],
                 ['D922', 'C331', 1.0],
                 ['D922', 'C934', 0.0]]

             uber.assignments()
             # Return list of assignments 
             # Example:  [['D32', 'C11'], ['D88', 'C934'], ['D922', 'C331']]

             uber.driver_assignment(driver)
             # Return customer assigned to driver or None

             uber.customer_assignment(customer)
             # Return driver assigned to customer or None

    """
    
    def __init__(self):
        self.model = LpProblem("Uber_Assignment", LpMinimize)

    def set_drivers(self, driver_list):
        # We omit some important error checking:...
        # "Do I have a valid list of driver IDs?"
        self.drivers = driver_list

    def set_customers(self, customer_list):
        self.customers = customer_list

    def set_travel_times(self, travel_time_dict):
        self.travel_times = travel_time_dict

    def build_model(self):
        self.init_flow_variables()
        self.init_objective_function()
        self.init_constraints()

    def init_flow_variables(self):
        self.X= {}
        for driver in self.drivers:
            for customer in self.customers:
                self.X[(driver,customer)] = LpVariable(f"X_{driver}_{customer}", cat='Binary')

    def init_objective_function(self):
        obj = ''
        for driver in self.drivers: 
            for customer in self.customers: 
                obj += self.travel_times[(driver,customer)] * self.X[(driver,customer)]
        self.model += obj, "Cost of Customer Driver Assignment"

    def init_constraints(self):
        for driver in self.drivers:
            const = None
            for customer in self.customers:
                const += self.X[(driver,customer)]
            self.model += const <= 1, f"driver_{driver}"

        for customer in self.customers:
            const = None
            for driver in self.drivers:
                const += self.X[(driver,customer)]
            self.model += const <= 1, f"customer_{customer}"

        num_assignments = min(len(self.drivers),len(self.customers))
        const = None
        for driver in self.drivers:
            for customer in self.customers:
                const += self.X[(driver,customer)]
        self.model += const == num_assignments, f"Number_of_Assignments"

    def print_model(self):
        print(self.model)

    def solve_model(self):
        self.model.solve(solver)
        
    def status(self):
        return LpStatus[self.model.status]
    
    def solution(self):
        solution = []
        for v in self.model.variables():
            match_vars = v.name.replace("X_","")
            assign_vars = match_vars.split("_")
            assign_vars.append(v.varValue)
            solution.append(assign_vars)
        return solution

    def assignments(self):
        assignments = []
        for v in self.model.variables():
            if v.varValue == 1.0:
                assignments.append(v.name.replace("X_","").split("_"))
        return assignments

    def drive_assignment(self, driver):
        # Return driver assigned or None
        assignments = self.assignments()
        for assignment in assignments:
            if(assignment [0] == driver):
                return assignment[1]
        return None

    def customer_assignment(self, customer):
        # Return driver assigned or None
        assignments = self.assignments()
        for assignment in assignments:
            if(assignment [1] == customer):
                return assignment[0]
        return None

    def objective_function_value(self):
        return value(self.model.objective)

In [3]:
uber = Uber()

In [4]:
uber.set_drivers(['D32', 'D8712', 'D922', 'D88'])
uber.set_customers(['C11', 'C934', 'C331'])
uber.set_travel_times({('D32', 'C11'): 587,
 ('D32', 'C331'): 1380,
 ('D32', 'C934'): 915,
 ('D8712', 'C11'): 1734,
 ('D8712', 'C331'): 693,
 ('D8712', 'C934'): 974,
 ('D88', 'C11'): 621,
 ('D88', 'C331'): 1230,
 ('D88', 'C934'): 830,
 ('D922', 'C11'): 1462,
 ('D922', 'C331'): 472,
 ('D922', 'C934'): 958})

In [5]:
uber.build_model()

In [6]:
uber.print_model()

Uber_Assignment:
MINIMIZE
587*X_D32_C11 + 1380*X_D32_C331 + 915*X_D32_C934 + 1734*X_D8712_C11 + 693*X_D8712_C331 + 974*X_D8712_C934 + 621*X_D88_C11 + 1230*X_D88_C331 + 830*X_D88_C934 + 1462*X_D922_C11 + 472*X_D922_C331 + 958*X_D922_C934 + 0
SUBJECT TO
driver_D32: X_D32_C11 + X_D32_C331 + X_D32_C934 <= 1

driver_D8712: X_D8712_C11 + X_D8712_C331 + X_D8712_C934 <= 1

driver_D922: X_D922_C11 + X_D922_C331 + X_D922_C934 <= 1

driver_D88: X_D88_C11 + X_D88_C331 + X_D88_C934 <= 1

customer_C11: X_D32_C11 + X_D8712_C11 + X_D88_C11 + X_D922_C11 <= 1

customer_C934: X_D32_C934 + X_D8712_C934 + X_D88_C934 + X_D922_C934 <= 1

customer_C331: X_D32_C331 + X_D8712_C331 + X_D88_C331 + X_D922_C331 <= 1

Number_of_Assignments: X_D32_C11 + X_D32_C331 + X_D32_C934 + X_D8712_C11
 + X_D8712_C331 + X_D8712_C934 + X_D88_C11 + X_D88_C331 + X_D88_C934
 + X_D922_C11 + X_D922_C331 + X_D922_C934 = 3

VARIABLES
0 <= X_D32_C11 <= 1 Integer
0 <= X_D32_C331 <= 1 Integer
0 <= X_D32_C934 <= 1 Integer
0 <= X_D8712_C11 <

In [7]:
uber.solve_model()

In [8]:
uber.solution()

[['D32', 'C11', 1.0],
 ['D32', 'C331', 0.0],
 ['D32', 'C934', 0.0],
 ['D8712', 'C11', 0.0],
 ['D8712', 'C331', 0.0],
 ['D8712', 'C934', 0.0],
 ['D88', 'C11', 0.0],
 ['D88', 'C331', 0.0],
 ['D88', 'C934', 1.0],
 ['D922', 'C11', 0.0],
 ['D922', 'C331', 1.0],
 ['D922', 'C934', 0.0]]

In [9]:
uber.assignments()

[['D32', 'C11'], ['D88', 'C934'], ['D922', 'C331']]

In [10]:
uber.customer_assignment('C934')

'D88'

In [11]:
uber.customer_assignment('11')

In [12]:
uber.status()

'Optimal'

In [13]:
uber.objective_function_value()

1889.0